<a href="https://colab.research.google.com/github/prometheus404/NLP_proj/blob/master/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NLP project

In [1]:
%pip install -U bitsandbytes

In [2]:
from llama_cpp import Llama
from tqdm import tqdm
from transformers import AutoTokenizer, pipeline, BitsAndBytesConfig
import requests
from collections import defaultdict
import json


In [3]:
# Load the model
#llm = Llama.from_pretrained(repo_id="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF", # repository name
#                            filename="Meta-Llama-3.1-8B-Instruct-Q8_0.gguf", # model file
#                            n_gpu_layers=-1, # use all GPU layers
#                            n_ctx=32768, # context size
#                            flash_attn=True, # use flash attention
#                            chat_format="llama-3", # chat format
#                            verbose=False)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)
mistral = pipeline(task="text-generation", model="mistralai/Mistral-7B-Instruct-v0.1", device_map="auto", model_kwargs={"quantization_config": quantization_config})


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


In [55]:
rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/dominion.txt').text
rulebook[:100]

'# Dominion\nYou are a monarch, like your parents before you - a ruler of a small pleasant kingdom of '

In [56]:
# Check that input is inside context window
tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")

tokens = tokenizer.encode(rulebook)
print(len(tokens))

8160


In [6]:
def generate_message(prompt, rulebook):
    return [
        {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": "Here is the rulebook:\n"+rulebook,
            },
    ]

In [52]:
import torch
def multiple_model_test(models, prompts, games, iterations):
    outputs = defaultdict(dict)

    for model_name,model,game,prompt_name,prompt,it in tqdm([(mn,m,f,pn,p,it) for f in game_names for (pn,p) in prompts for (mn,m) in models for it in range(iterations)]):
        rulebook = requests.get('https://raw.githubusercontent.com/prometheus404/NLP_proj/refs/heads/master/rules/texts/'+game+'.txt').text
        out = model(generate_message(prompt, rulebook))
        outputs[model_name][game+'-'+prompt_name+'-'+str(it)] = out[0]['generated_text'][-1]['content']
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


    with open('extraction.out','w') as f:
        json.dump(dict(outputs),f)

    return outputs

# Rule extraction
1. Give the model a rulebook and prompt it to explain the game in simple, conversational terms to a child or other audiences. -> tree decomposition to test how well the model did
2. Test the ability of the model to find analogies of rules (?)
3. Test the ability to extract if-then rules (?)
4. Organize the rules of into a hierarchy: top-level objectives, mid-level phases, low-level actions. (?) (look into the paper)

In [ ]:
prompts = [("kid", """You are a friendly tutor explaining board games to a 7‑year‑old. Summarize the game in plain language, using short sentences and with fun tone. Include:
                - Goal of the game
                - How a player wins
                - What a turn looks like
                - Exceptions to standard rules

                The user will give you a text file with the rulebook you need to explain.
                the output should not be too long. All rules must be present in your explanation"""),
           ("analogies", """The user will give you a text file with the rulebook you need to explain
           the output should not be too long. All rules must be present in your explanation.
           Explain the rules to a child by comparing it to something they already know (e.g., “like a treasure hunt” or “like building a LEGO city”).
           se the rulebook to keep the analogy accurate, and end with a one‑sentence “what you try to achieve” statement.\n"""),
]
game_names = [ 'catan', 'power_grid','ticket_to_ride',]
models = [("mistral", mistral)]
iterations = 1

multiple_model_test(models, prompts, game_names, iterations)

 50%|█████     | 3/6 [02:25<02:07, 42.66s/it]Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [51]:
out = mistral(generate_message('explain the rulebooks given to you in a coincise way', rulebook))


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


In [50]:
print(out[0]['generated_text'][-1]['content'])

 "North American Trains" is a card game where players compete to score the highest number of points. The game includes components such as a rules booklet, game board, plastic trains, scoring markers, ticket cards, and train cards.

To set up the game, each player takes a set of 45 trains and one scoring marker. The board is placed in the center of the table and the remaining cards are shuffled and dealt to the players.

The game is played clockwise around the table, with each player taking one turn at a time. On their turn, a player can draw train cards, claim a route, or draw tickets.

To claim a route, a player must play a set of train cards that match the color of the route. The points earned for a route depend on its length and whether it is completed successfully.

To draw tickets, a player can draw three tickets from the top of the ticket deck. They must keep at least one of the tickets they draw and any returned tickets are placed at the bottom of the deck.

The game ends when a

## Error detection

 Each rulebook is edited by inserting a set of 5 errors each of increasing difficulty:
 - level 1 -> **unsolvable** mechanic that uses piece not presents in the game (every time you play a card you can steal a warrior token from every opponent but there is no way to obtain a warrior token)
- level 2 -> **unsolvable** (in one line states you can draw two cards, in another one that you can draw only one)
- level 3 -> **incoherent** and hardlocks the game (you cannot play train if you do not have train on the map but on another line clearly states you start with an empty map)
- level 4 -> **coherent** but obviously gamebreaking (draw infinite cards each turn)
- level 5 -> **coherent** but very unbalanced (the first player can play two turns)

In [ ]:
prompts = ["""test"""]
game_names = ['ticket_to_ride', 'dominion', 'catan', 'power_grid']
models = [llm]
iterations = 5

multiple_model_test(models, prompts, game_names, iterations)

# Game classification
Give the model a rulebook and ask it to classify the mechanics, evaluate the complexity, suggests the perfect number of players and estimate the duration